# Cointegration Strategy Testing Framework

This notebook tests baskets for cointegration sustainability and mean reversion speed.

## Testing Pipeline:
1. Load price data from parquet files
2. Generate candidate baskets (or load existing)
3. Test initial cointegration
4. Filter by sustainability across time periods
5. Filter by mean reversion speed
6. Output validated baskets for strategy deployment

## Testing Principles:
- **No redundancy**: Each test serves a unique purpose
- **Trading-focused**: All metrics tied to trading outcomes
- **Clear rationale**: Documented filter thresholds

In [ ]:
import sys
from pathlib import Path
import logging
import pandas as pd
import numpy as np
from datetime import datetime, timezone

# Add project root to path
project_root = Path().resolve().parent.parent.parent.parent
sys.path.insert(0, str(project_root))

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Import cointegration modules
from hypothesis_testing.cointegration.data_loader import load_price_data
from hypothesis_testing.cointegration.basket_generator import generate_baskets_clustering, compute_spread
from hypothesis_testing.cointegration.utils_parallel import test_baskets_cointegration_parallel
from hypothesis_testing.cointegration.filter_sustainability import filter_baskets_sustainability
from hypothesis_testing.cointegration.filter_mean_reversion import filter_baskets_mean_reversion

logger.info("Imports completed")

## Configuration

Set data paths, timeframes, and filter thresholds.

In [ ]:
# Data configuration
DATA_DIR = Path("../../hku-data/test_data")  # Relative to notebook location
TIMEFRAME = '1h'  # '1h' for index or '15m' for mark
PRICE_TYPE = 'index'  # 'index' or 'mark'
SYMBOLS = None  # None = auto-discover, or specify: ["BTC", "ETH", "SOL"]

# Basket generation
MIN_BASKET_SIZE = 2
MAX_BASKET_SIZE = 4
N_CLUSTERS = 5

# Sustainability filter thresholds
PERSISTENCE_THRESHOLD = 0.7  # 70% of windows/periods must pass
WINDOW_DAYS = 90
STEP_DAYS = 30
PERIOD_DAYS = 90

# Mean reversion filter thresholds
HALF_LIFE_THRESHOLD_DAYS = 30.0  # Maximum half-life in days

# Parallel processing
MAX_WORKERS = None  # None = use CPU count

# Bars per day (depends on timeframe)
BARS_PER_DAY = 24 if TIMEFRAME == '1h' else 96

logger.info(f"Configuration: timeframe={TIMEFRAME}, price_type={PRICE_TYPE}, "
           f"persistence_threshold={PERSISTENCE_THRESHOLD:.0%}, "
           f"half_life_threshold={HALF_LIFE_THRESHOLD_DAYS} days")

## Step 1: Load Price Data

Load historical price data from parquet files.

In [ ]:
price_data = load_price_data(
    data_dir=DATA_DIR,
    symbols=SYMBOLS,
    timeframe=TIMEFRAME,
    price_type=PRICE_TYPE,
    max_workers=MAX_WORKERS
)

logger.info(f"Loaded price data: {len(price_data)} timestamps, {len(price_data.columns)} symbols")
logger.info(f"Date range: {price_data.index.min()} to {price_data.index.max()}")
price_data.head()

## Step 2: Generate Candidate Baskets

Generate candidate baskets using clustering (or load existing baskets).

In [ ]:
# Extract base symbols (remove _close suffix)
base_symbols = [col.replace('_close', '') for col in price_data.columns]

candidate_baskets = generate_baskets_clustering(
    price_data=price_data,
    n_clusters=N_CLUSTERS,
    min_basket_size=MIN_BASKET_SIZE,
    max_basket_size=MAX_BASKET_SIZE,
    parallel=True,
    max_workers=MAX_WORKERS
)

logger.info(f"Generated {len(candidate_baskets)} candidate baskets")
print(f"Sample baskets: {candidate_baskets[:5]}")

## Step 3: Test Initial Cointegration

Test all candidate baskets for cointegration using Johansen test (multiprocessed).

In [ ]:
cointegrated_baskets = test_baskets_cointegration_parallel(
    price_data=price_data,
    candidate_baskets=candidate_baskets,
    max_workers=MAX_WORKERS,
    batch_size=100
)

logger.info(f"Found {len(cointegrated_baskets)} cointegrated baskets out of {len(candidate_baskets)} candidates")
print(f"Cointegration rate: {len(cointegrated_baskets)/len(candidate_baskets):.1%}")

## Step 4: Filter by Sustainability

Test cointegration persistence across rolling windows and discrete periods. 
This ensures the cointegration relationship is stable over time (not just in-sample).

In [ ]:
sustainable_baskets = filter_baskets_sustainability(
    baskets=cointegrated_baskets,
    price_data=price_data,
    persistence_threshold=PERSISTENCE_THRESHOLD,
    window_days=WINDOW_DAYS,
    step_days=STEP_DAYS,
    period_days=PERIOD_DAYS,
    bars_per_day=BARS_PER_DAY,
    use_rolling=True,
    use_discrete=True,
    max_workers=MAX_WORKERS
)

logger.info(f"Filtered to {len(sustainable_baskets)} sustainable baskets "
           f"(from {len(cointegrated_baskets)} cointegrated)")

# Display sample results
if sustainable_baskets:
    sample = sustainable_baskets[0]
    print(f"\nSample sustainable basket: {sample['basket']}")
    print(f"Rolling windows persistence: {sample['sustainability']['rolling_windows']['persistence_ratio']:.1%}")
    print(f"Discrete periods persistence: {sample['sustainability']['discrete_periods']['persistence_ratio']:.1%}")

## Step 5: Filter by Mean Reversion Speed

Filter for baskets that mean revert quickly (important for trading performance).
This tests whether the spread reverts fast enough to be profitable.

In [ ]:
fast_mean_reverting_baskets = filter_baskets_mean_reversion(
    baskets=sustainable_baskets,
    half_life_threshold_days=HALF_LIFE_THRESHOLD_DAYS,
    bars_per_day=BARS_PER_DAY,
    max_workers=MAX_WORKERS
)

logger.info(f"Filtered to {len(fast_mean_reverting_baskets)} fast mean-reverting baskets "
           f"(from {len(sustainable_baskets)} sustainable)")

# Display sample results
if fast_mean_reverting_baskets:
    sample = fast_mean_reverting_baskets[0]
    print(f"\nSample fast mean-reverting basket: {sample['basket']}")
    print(f"Half-life: {sample['mean_reversion']['half_life_days']:.1f} days")
    print(f"ADF p-value: {sample['mean_reversion']['adf_p_value']:.4f}")
    print(f"Is stationary: {sample['mean_reversion']['is_stationary']}")

## Step 6: Save Validated Baskets

Save the proven baskets for strategy deployment.

In [ ]:
import json

# Prepare baskets for saving (convert numpy arrays to lists)
validated_baskets_output = []
for basket_result in fast_mean_reverting_baskets:
    output = {
        'basket': basket_result['basket'],
        'eigenvector': basket_result['eigenvector'].tolist(),
        'johansen_p_value': basket_result['johansen_result']['p_value'],
        'johansen_trace_stat': float(basket_result['johansen_result']['trace_stat']),
        'sustainability_rolling': basket_result['sustainability']['rolling_windows']['persistence_ratio'],
        'sustainability_discrete': basket_result['sustainability']['discrete_periods']['persistence_ratio'],
        'half_life_days': float(basket_result['mean_reversion']['half_life_days']),
        'adf_p_value': float(basket_result['mean_reversion']['adf_p_value']),
        'is_stationary': basket_result['mean_reversion']['is_stationary']
    }
    validated_baskets_output.append(output)

# Save to JSON
output_file = Path("validated_baskets.json")
with open(output_file, 'w') as f:
    json.dump({
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'config': {
            'timeframe': TIMEFRAME,
            'price_type': PRICE_TYPE,
            'persistence_threshold': PERSISTENCE_THRESHOLD,
            'half_life_threshold_days': HALF_LIFE_THRESHOLD_DAYS
        },
        'baskets': validated_baskets_output
    }, f, indent=2)

logger.info(f"Saved {len(validated_baskets_output)} validated baskets to {output_file}")

# Display summary
print(f"\n{'='*60}")
print("VALIDATION SUMMARY")
print(f"{'='*60}")
print(f"Total candidate baskets: {len(candidate_baskets)}")
print(f"Cointegrated baskets: {len(cointegrated_baskets)} ({len(cointegrated_baskets)/len(candidate_baskets):.1%})")
print(f"Sustainable baskets: {len(sustainable_baskets)} ({len(sustainable_baskets)/len(cointegrated_baskets):.1%} of cointegrated)")
print(f"Fast mean-reverting baskets: {len(fast_mean_reverting_baskets)} ({len(fast_mean_reverting_baskets)/len(sustainable_baskets):.1%} of sustainable)")
print(f"{'='*60}")
print(f"\nValidated baskets ready for strategy deployment!")
print(f"Output file: {output_file}")